In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import rankdata

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from category_encoders import TargetEncoder


# =========================================================
# 0. Config
# =========================================================

TRAIN_PATH = "../data/train.csv"
TEST_PATH = "../data/test.csv"
SUBMISSION_PATH = "../data/sample_submission.csv"
TARGET = "임신 성공 여부"

FOLD_SEED = 1234
N_SPLITS = 5

POS_WEIGHT = 190123 / 66228

SAVE_DIR = Path("../oof_preds/combo_te_v1_s10_seed1234")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

SUB_DIR = Path("submissions/seed_ensemble")
SUB_DIR.mkdir(parents=True, exist_ok=True)



def data_preprocessing(df):
    df = df.copy()
    time_cols = [
        '임신 시도 또는 마지막 임신 경과 연수',
        '난자 해동 경과일',
        '난자 혼합 경과일',
        '배아 이식 경과일',
        '배아 해동 경과일'
    ]

    for col in time_cols:
        df[f'{col}_performed'] = (
            df[col].notnull()
        ).astype(int)


    df = df.fillna(0)
    infertility_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 남성 요인',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    male_cols = [
        '불임 원인 - 남성 요인',
        '불임 원인 - 정자 농도',
        '불임 원인 - 정자 운동성',
        '불임 원인 - 정자 형태',
        '불임 원인 - 정자 면역학적 요인'
    ]

    female_cols = [
        '불임 원인 - 난관 질환',
        '불임 원인 - 배란 장애',
        '불임 원인 - 여성 요인',
        '불임 원인 - 자궁경부 문제',
        '불임 원인 - 자궁내막증'
    ]

    df['불임원인_총개수'] = df[infertility_cols].sum(axis=1)

    df['남성_원인_수'] = df[male_cols].sum(axis=1)

    df['여성_원인_수'] = df[female_cols].sum(axis=1)

    df['남녀_복합_원인'] = (
        (df['남성_원인_수'] > 0) &
        (df['여성_원인_수'] > 0)
    ).astype(int)

    df['원인불명'] = (
        df['불임원인_총개수'] == 0
    ).astype(int)

    count_cols = [
        '총 시술 횟수',
        'IVF 시술 횟수',
        'DI 시술 횟수',
        '총 임신 횟수',
        'IVF 임신 횟수',
        'DI 임신 횟수',
        '총 출산 횟수',
        'IVF 출산 횟수',
        'DI 출산 횟수',
        '클리닉 내 총 시술 횟수'
    ]

    count_map = {
        '0회': 0,
        '1회': 1,
        '2회': 2,
        '3회': 3,
        '4회': 4,
        '5회': 5,
        '6회 이상': 6,
    }

    for col in count_cols:
        df[col] = df[col].map(count_map).astype(float)

    df['고령여부'] = df['시술 당시 나이'].isin([
        '만38-39세',
        '만40-42세',
        '만43-44세',
        '만45-50세'
    ]).astype(int)


    df['배아_생성률'] = np.where(
        df['혼합된 난자 수'] == 0,
        0,
        df['총 생성 배아 수'] / df['혼합된 난자 수']
    )

    # 2. 배아 이식 효율
    df['배아_이식률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['이식된 배아 수'] / df['총 생성 배아 수']
    )

    # 3. 배아 냉동 비율
    df['배아_냉동률'] = np.where(
        df['총 생성 배아 수'] == 0,
        0,
        df['저장된 배아 수'] / df['총 생성 배아 수']
    )
    df['IVF_임신성공률'] = np.where(
        df['IVF 시술 횟수'] == 0,
        0,
        df['IVF 임신 횟수'] / df['IVF 시술 횟수']
    )

    df['DI_임신성공률'] = np.where(
        df['DI 시술 횟수'] == 0,
        0,
        df['DI 임신 횟수'] / df['DI 시술 횟수']
    )

    df['고령_난자수_interaction'] = (
        df['고령여부'] *
        df['수집된 신선 난자 수']
    )

    df['배아이식_수행여부'] = (
        df['이식된 배아 수'] > 0
    ).astype(int)

    df['배아_이식_집중도'] = np.where(
        (df['이식된 배아 수'] + df['저장된 배아 수']) == 0,
        0,
        df['이식된 배아 수'] /
        (
            df['이식된 배아 수'] +
            df['저장된 배아 수']
        )
    )
    df["배아 생성 주요 이유"] = (
        df["배아 생성 주요 이유"]
        .astype(str)
        .astype("category")
    )
    df["특정 시술 유형"] = (
        df["특정 시술 유형"]
        .astype(str)
        .astype("category")
    )

    # 고령 × 이식 배아 수
    df['고령_배아이식'] = (
        df['고령여부'] *
        df['이식된 배아 수']
    )

    # 고령 × 총 생성 배아 수
    df['고령_배아생성'] = (
        df['고령여부'] *
        df['총 생성 배아 수']
    )

    # 고령 × 저장 배아 수
    df['고령_배아저장'] = (
        df['고령여부'] *
        df['저장된 배아 수']
    )

    # 고령 × 미세주입 난자 수
    df['고령_미세주입난자'] = (
        df['고령여부'] *
        df['미세주입된 난자 수']
    )

    df['출산_임신_전환율'] = np.where(
        df['총 임신 횟수'] == 0,
        0,
        df['총 출산 횟수'] / df['총 임신 횟수']
    )

    df['클리닉_집중도'] = np.where(
        df['총 시술 횟수'] == 0,
        0,
        df['클리닉 내 총 시술 횟수'] / df['총 시술 횟수']
    )

    df['첫_시술_여부'] = (
        df['총 시술 횟수'] == 0
    ).astype(int)



    binary_keywords = [
        "코드", "나이", "유형", "여부", "원인", "이유", "횟수", "출처"
    ]

    binary_cols = [
        col for col in df.columns
        if any(keyword in col for keyword in binary_keywords)
    ]

    df[binary_cols] = df[binary_cols].astype('category')

    object_cols = df.select_dtypes(include="object").columns

    df[object_cols] = df[object_cols].astype(str)

    cat_cols = df.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    for col in cat_cols:
        if col != TARGET:
            df[col] = df[col].astype(str)

    drop_cols = [
        '배아이식_수행여부'
    ]
    df = df.drop(
        columns=drop_cols
        )
    return df

In [2]:
# =========================================================
# 1. Utility
# =========================================================

def rank01(pred):
    return rankdata(pred) / len(pred)


def add_combo_columns(X):
    X = X.copy()

    # Combo TE v1 only
    combo_pairs = [
        ("시술 당시 나이", "난자 출처"),
        ("시술 당시 나이", "정자 출처"),
        ("시술 당시 나이", "시술 유형"),
        ("시술 당시 나이", "특정 시술 유형"),
        ("시술 유형", "난자 출처"),
        ("시술 유형", "정자 출처"),
        ("특정 시술 유형", "난자 출처"),
        ("특정 시술 유형", "정자 출처"),
        ("난자 출처", "정자 출처"),
        ("배란 유도 유형", "시술 당시 나이"),
    ]

    combo_cols = []

    for col1, col2 in combo_pairs:
        if col1 in X.columns and col2 in X.columns:
            new_col = f"{col1}_{col2}_combo"
            X[new_col] = X[col1].astype(str) + "_" + X[col2].astype(str)
            combo_cols.append(new_col)

    return X, combo_cols


def add_oof_target_encoding(X, y, cols, n_splits=5, smoothing=10, random_state=1234):
    X = X.copy().reset_index(drop=True)
    y = y.astype(int).reset_index(drop=True)

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state
    )

    te_features = pd.DataFrame(index=X.index)

    for col in cols:
        te_features[f"{col}_TE"] = 0.0

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        print(f"Target Encoding Fold {fold}")

        X_tr = X.iloc[train_idx]
        X_val = X.iloc[val_idx]
        y_tr = y.iloc[train_idx]

        encoder = TargetEncoder(
            cols=cols,
            smoothing=smoothing
        )

        encoder.fit(X_tr[cols], y_tr)
        encoded_val = encoder.transform(X_val[cols])

        for col in cols:
            te_features.loc[val_idx, f"{col}_TE"] = encoded_val[col].values

    final_encoder = TargetEncoder(
        cols=cols,
        smoothing=smoothing
    )

    final_encoder.fit(X[cols], y)

    X_te = pd.concat([X, te_features], axis=1)

    return X_te, final_encoder


def make_cat_params(kind="main", seed=42):
    if kind == "main":
        return dict(
            iterations=2000,
            learning_rate=0.02498214961001344,
            depth=8,
            l2_leaf_reg=18.591182129683194,
            random_strength=0.32969640414889206,
            bagging_temperature=4.535604806522509,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "shallow":
        return dict(
            iterations=2500,
            learning_rate=0.02,
            depth=5,
            l2_leaf_reg=8,
            random_strength=1.5,
            bagging_temperature=2,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    if kind == "random":
        return dict(
            iterations=2500,
            learning_rate=0.022,
            depth=7,
            l2_leaf_reg=15,
            random_strength=8,
            bagging_temperature=8,
            loss_function="Logloss",
            eval_metric="AUC",
            random_seed=seed,
            verbose=100,
            class_weights=[1, POS_WEIGHT],
            allow_writing_files=False,
        )

    raise ValueError(f"Unknown CatBoost kind: {kind}")

In [3]:
# =========================================================
# 2. Load & preprocess
# =========================================================

train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
submission = pd.read_csv(SUBMISSION_PATH)

X_raw = train_raw.drop(columns=[TARGET])
y = train_raw[TARGET].astype(int).reset_index(drop=True)
X_test_raw = test_raw.copy()

id_cols = [col for col in X_raw.columns if "ID" in col.upper()]

X_raw = X_raw.drop(columns=id_cols, errors="ignore")
X_test_raw = X_test_raw.drop(columns=id_cols, errors="ignore")

# 기존 champion data_preprocessing 함수 사용
X = data_preprocessing(X_raw)
X_test = data_preprocessing(X_test_raw)

X_combo, combo_cols = add_combo_columns(X)
X_test_combo, _ = add_combo_columns(X_test)

base_te_cols = [
    "시술 시기 코드",
    "시술 유형",
    "특정 시술 유형",
    "배란 유도 유형",
    "난자 출처",
    "정자 출처",
    "배아 생성 주요 이유",
    "시술 당시 나이",
]

base_te_cols = [col for col in base_te_cols if col in X_combo.columns]
te_cols = base_te_cols + combo_cols

print("base_te_cols:", base_te_cols)
print("combo_cols:", combo_cols)
print("te_cols count:", len(te_cols))

X_te_combo, te_encoder_combo = add_oof_target_encoding(
    X_combo,
    y,
    cols=te_cols,
    n_splits=N_SPLITS,
    smoothing=10,
    random_state=FOLD_SEED
)

# test TE
test_te_values = te_encoder_combo.transform(X_test_combo[te_cols])

for col in te_cols:
    X_test_combo[f"{col}_TE"] = test_te_values[col].values

# combo 문자열 원본 제거
X_te_combo = X_te_combo.drop(columns=combo_cols, errors="ignore")
X_test_te_combo = X_test_combo.drop(columns=combo_cols, errors="ignore")

# 컬럼 정렬
X_test_te_combo = X_test_te_combo[X_te_combo.columns]

X_stack = X_te_combo.copy().reset_index(drop=True)
X_test_stack = X_test_te_combo.copy().reset_index(drop=True)
y_stack = y.reset_index(drop=True)

cat_cols = X_stack.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in cat_cols:
    X_stack[col] = X_stack[col].astype(str)
    X_test_stack[col] = X_test_stack[col].astype(str)

print("X_stack:", X_stack.shape)
print("X_test_stack:", X_test_stack.shape)
print("y_stack:", y_stack.shape)
print("cat_cols:", len(cat_cols))
print("combo cols remaining:", [c for c in X_stack.columns if c.endswith("_combo")])

assert list(X_stack.columns) == list(X_test_stack.columns)
assert len(X_stack) == len(y_stack)
assert len(X_test_stack) == len(submission)

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=FOLD_SEED
)

base_te_cols: ['시술 시기 코드', '시술 유형', '특정 시술 유형', '배란 유도 유형', '난자 출처', '정자 출처', '배아 생성 주요 이유', '시술 당시 나이']
combo_cols: ['시술 당시 나이_난자 출처_combo', '시술 당시 나이_정자 출처_combo', '시술 당시 나이_시술 유형_combo', '시술 당시 나이_특정 시술 유형_combo', '시술 유형_난자 출처_combo', '시술 유형_정자 출처_combo', '특정 시술 유형_난자 출처_combo', '특정 시술 유형_정자 출처_combo', '난자 출처_정자 출처_combo', '배란 유도 유형_시술 당시 나이_combo']
te_cols count: 18
Target Encoding Fold 1
Target Encoding Fold 2
Target Encoding Fold 3
Target Encoding Fold 4
Target Encoding Fold 5
X_stack: (256351, 110)
X_test_stack: (90067, 110)
y_stack: (256351,)
cat_cols: 54
combo cols remaining: []


In [4]:
# =========================================================
# 3. CatBoost train functions
# =========================================================

def train_cat_seed_ensemble(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    seeds,
    kind="main",
    name="main_cat"
):
    seed_oof_list = []
    seed_test_list = []
    score_rows = []

    for seed in seeds:
        print(f"\n================ {name} seed {seed} ================")

        oof = np.zeros(len(X_stack))
        test_pred = np.zeros(len(X_test_stack))

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
            print(f"\n{name} seed {seed} / Fold {fold}")

            X_tr = X_stack.iloc[tr_idx].copy()
            X_val = X_stack.iloc[val_idx].copy()
            y_tr = y_stack.iloc[tr_idx]
            y_val = y_stack.iloc[val_idx]

            model = CatBoostClassifier(**make_cat_params(kind=kind, seed=seed))

            early_stop = 100 if kind == "main" else 150

            model.fit(
                X_tr,
                y_tr,
                cat_features=cat_cols,
                eval_set=(X_val, y_val),
                early_stopping_rounds=early_stop,
                verbose=100
            )

            val_pred = model.predict_proba(X_val)[:, 1]
            oof[val_idx] = val_pred

            fold_auc = roc_auc_score(y_val, val_pred)
            print(f"{name} seed {seed} Fold {fold} AUC:", fold_auc)

            score_rows.append({
                "model": name,
                "seed": seed,
                "fold": fold,
                "auc": fold_auc
            })

            test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        seed_auc = roc_auc_score(y_stack, oof)
        print(f"\n{name} seed {seed} OOF AUC:", seed_auc)

        score_rows.append({
            "model": name,
            "seed": seed,
            "fold": "OOF",
            "auc": seed_auc
        })

        seed_oof_list.append(oof)
        seed_test_list.append(test_pred)

        np.save(SAVE_DIR / f"{name}_seed{seed}_oof.npy", oof)
        np.save(SAVE_DIR / f"{name}_seed{seed}_test.npy", test_pred)

    final_oof = np.mean(seed_oof_list, axis=0)
    final_test = np.mean(seed_test_list, axis=0)

    final_auc = roc_auc_score(y_stack, final_oof)
    print(f"\n{name} seed ensemble OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": "ensemble",
        "fold": "OOF",
        "auc": final_auc
    })

    return final_oof, final_test, pd.DataFrame(score_rows)


def train_cat_single(
    X_stack,
    X_test_stack,
    y_stack,
    cat_cols,
    skf,
    kind="shallow",
    name="shallow_cat"
):
    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ {name} Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        model = CatBoostClassifier(**make_cat_params(kind=kind, seed=42))

        early_stop = 150 if kind == "shallow" else 200

        model.fit(
            X_tr,
            y_tr,
            cat_features=cat_cols,
            eval_set=(X_val, y_val),
            early_stopping_rounds=early_stop,
            verbose=100
        )

        val_pred = model.predict_proba(X_val)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print(f"{name} Fold {fold} AUC:", fold_auc)

        score_rows.append({
            "model": name,
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_stack)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"{name}_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"{name}_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print(f"\n{name} OOF AUC:", final_auc)

    score_rows.append({
        "model": name,
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)

In [5]:
# =========================================================
# 4. CatBoost models
# =========================================================

main_cat_oof, main_cat_test_pred, main_score_df = train_cat_seed_ensemble(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    seeds=[42, 77, 2024],
    kind="main",
    name="main_cat"
)

np.save(SAVE_DIR / "main_cat_oof.npy", main_cat_oof)
np.save(SAVE_DIR / "main_cat_test_pred.npy", main_cat_test_pred)


shallow_cat_oof, shallow_cat_test_pred, shallow_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="shallow",
    name="shallow_cat"
)

np.save(SAVE_DIR / "shallow_cat_oof.npy", shallow_cat_oof)
np.save(SAVE_DIR / "shallow_cat_test_pred.npy", shallow_cat_test_pred)


random_cat_oof, random_cat_test_pred, random_score_df = train_cat_single(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    cat_cols=cat_cols,
    skf=skf,
    kind="random",
    name="random_cat"
)

np.save(SAVE_DIR / "random_cat_oof.npy", random_cat_oof)
np.save(SAVE_DIR / "random_cat_test_pred.npy", random_cat_test_pred)


================ main_cat seed 42 ================

main_cat seed 42 / Fold 1
0:	test: 0.7326893	best: 0.7326893 (0)	total: 4.37s	remaining: 2h 25m 33s
100:	test: 0.7411347	best: 0.7411347 (100)	total: 25s	remaining: 7m 50s
200:	test: 0.7428194	best: 0.7428216 (196)	total: 39.7s	remaining: 5m 55s
300:	test: 0.7432711	best: 0.7432711 (300)	total: 53.9s	remaining: 5m 4s
400:	test: 0.7435205	best: 0.7435278 (386)	total: 1m 8s	remaining: 4m 34s
500:	test: 0.7435852	best: 0.7435863 (497)	total: 1m 22s	remaining: 4m 5s
600:	test: 0.7437236	best: 0.7437236 (600)	total: 1m 51s	remaining: 4m 18s
700:	test: 0.7438513	best: 0.7438513 (700)	total: 2m 5s	remaining: 3m 51s
800:	test: 0.7439769	best: 0.7439787 (798)	total: 2m 18s	remaining: 3m 27s
900:	test: 0.7440214	best: 0.7440288 (899)	total: 2m 31s	remaining: 3m 4s
1000:	test: 0.7439933	best: 0.7440321 (915)	total: 2m 46s	remaining: 2m 45s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.7440320609
bestIteration = 915

Shrin

In [6]:
# =========================================================
# 5. XGB OOF + test
# =========================================================

def train_xgb_oof_test(X_stack, X_test_stack, y_stack, skf):
    numeric_features = X_stack.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_stack.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])

    oof = np.zeros(len(X_stack))
    test_pred = np.zeros(len(X_test_stack))
    score_rows = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_stack, y_stack), 1):
        print(f"\n================ XGB Fold {fold} ================")

        X_tr = X_stack.iloc[tr_idx].copy()
        X_val = X_stack.iloc[val_idx].copy()
        y_tr = y_stack.iloc[tr_idx]
        y_val = y_stack.iloc[val_idx]

        X_tr_trans = preprocessor.fit_transform(X_tr)
        X_val_trans = preprocessor.transform(X_val)
        X_test_trans = preprocessor.transform(X_test_stack)

        model = XGBClassifier(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=5,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=42,
            scale_pos_weight=POS_WEIGHT,
            tree_method="hist",
            n_jobs=-1
        )

        model.fit(
            X_tr_trans,
            y_tr.to_numpy().ravel(),
            eval_set=[(X_val_trans, y_val.to_numpy().ravel())],
            verbose=False
        )

        val_pred = model.predict_proba(X_val_trans)[:, 1]
        oof[val_idx] = val_pred

        fold_auc = roc_auc_score(y_val, val_pred)
        print("XGB Fold AUC:", fold_auc)

        score_rows.append({
            "model": "xgb",
            "seed": 42,
            "fold": fold,
            "auc": fold_auc
        })

        test_pred += model.predict_proba(X_test_trans)[:, 1] / skf.n_splits

        np.save(SAVE_DIR / f"xgb_partial_oof_fold{fold}.npy", oof)
        np.save(SAVE_DIR / f"xgb_partial_test_fold{fold}.npy", test_pred)

    final_auc = roc_auc_score(y_stack, oof)
    print("\nXGB OOF AUC:", final_auc)

    score_rows.append({
        "model": "xgb",
        "seed": 42,
        "fold": "OOF",
        "auc": final_auc
    })

    return oof, test_pred, pd.DataFrame(score_rows)


xgb_oof, xgb_test_pred, xgb_score_df = train_xgb_oof_test(
    X_stack=X_stack,
    X_test_stack=X_test_stack,
    y_stack=y_stack,
    skf=skf
)

np.save(SAVE_DIR / "xgb_oof.npy", xgb_oof)
np.save(SAVE_DIR / "xgb_test_pred.npy", xgb_test_pred)


================ XGB Fold 1 ================
XGB Fold AUC: 0.7431218018454804

================ XGB Fold 2 ================
XGB Fold AUC: 0.7389192483040379

================ XGB Fold 3 ================
XGB Fold AUC: 0.7356902645708292

================ XGB Fold 4 ================
XGB Fold AUC: 0.7374861650353047

================ XGB Fold 5 ================
XGB Fold AUC: 0.7359988805198139

XGB OOF AUC: 0.7380293600861654


In [7]:
# =========================================================
# 6. OOF validation: rank blend
# =========================================================
main_cat_rank_oof = rank01(main_cat_oof)
shallow_cat_rank_oof = rank01(shallow_cat_oof)
random_cat_rank_oof = rank01(random_cat_oof)
xgb_rank_oof = rank01(xgb_oof)

final_oof_seed1234 = (
    0.44 * main_cat_rank_oof +
    0.08 * shallow_cat_rank_oof +
    0.35 * random_cat_rank_oof +
    0.13 * xgb_rank_oof
)

final_oof_auc = roc_auc_score(y_stack, final_oof_seed1234)

print("\n====================")
print("Seed1234 Main Cat OOF:", roc_auc_score(y_stack, main_cat_oof))
print("Seed1234 Shallow Cat OOF:", roc_auc_score(y_stack, shallow_cat_oof))
print("Seed1234 Random Cat OOF:", roc_auc_score(y_stack, random_cat_oof))
print("Seed1234 XGB OOF:", roc_auc_score(y_stack, xgb_oof))
print("Seed1234 Final Rank Blend OOF:", final_oof_auc)

np.save(SAVE_DIR / "final_oof_seed1234.npy", final_oof_seed1234)
np.save(SAVE_DIR / "y_stack.npy", y_stack.to_numpy())


score_df = pd.concat(
    [main_score_df, shallow_score_df, random_score_df, xgb_score_df],
    ignore_index=True
)

score_df.to_csv(SAVE_DIR / "fold_oof_scores.csv", index=False)

summary_df = pd.DataFrame([
    {
        "fold_seed": FOLD_SEED,
        "feature_set": "combo_te_v1_s10",
        "main_cat_oof": roc_auc_score(y_stack, main_cat_oof),
        "shallow_cat_oof": roc_auc_score(y_stack, shallow_cat_oof),
        "random_cat_oof": roc_auc_score(y_stack, random_cat_oof),
        "xgb_oof": roc_auc_score(y_stack, xgb_oof),
        "final_rank_blend_oof": final_oof_auc,
        "weights": "main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13"
    }
])

summary_df.to_csv(SAVE_DIR / "summary.csv", index=False)
display(summary_df)


Seed1234 Main Cat OOF: 0.7403288816747414
Seed1234 Shallow Cat OOF: 0.7402502816533824
Seed1234 Random Cat OOF: 0.7401278934029106
Seed1234 XGB OOF: 0.7380293600861654
Seed1234 Final Rank Blend OOF: 0.7403779062281844


,fold_seed,feature_set,main_cat_oof,shallow_cat_oof,random_cat_oof,xgb_oof,final_rank_blend_oof,weights
0,1234,combo_te_v1_s10,0.740329,0.74025,0.740128,0.738029,0.740378,main 0.44 / shallow 0.08 / random 0.35 / xgb 0.13


In [9]:
# =========================================================
# 7. Test prediction: seed1234 submission
# =========================================================
random_cat_test_pred = np.load(SAVE_DIR / "random_cat_test_pred.npy")
main_cat_test_pred = np.load(SAVE_DIR / "main_cat_test_pred.npy")
random_cat_test_pred = np.load(SAVE_DIR / "random_cat_test_pred.npy")
xgb_test_pred = np.load(SAVE_DIR / "xgb_test_pred.npy")

main_cat_rank_test = rank01(main_cat_test_pred)
shallow_cat_rank_test = rank01(shallow_cat_test_pred)
random_cat_rank_test = rank01(random_cat_test_pred)
xgb_rank_test = rank01(xgb_test_pred)

final_pred_seed1234 = (
    0.44 * main_cat_rank_test +
    0.08 * shallow_cat_rank_test +
    0.35 * random_cat_rank_test +
    0.13 * xgb_rank_test
)

assert len(final_pred_seed1234) == len(submission)
assert np.isfinite(final_pred_seed1234).all()
assert final_pred_seed1234.min() >= 0
assert final_pred_seed1234.max() <= 1

pred_col = submission.columns[-1]

submission_seed1234 = submission.copy()
submission_seed1234[pred_col] = final_pred_seed1234

submission_seed1234.to_csv(
    SUB_DIR / "submission_combo_te_v1_rank_seed1234.csv",
    index=False
)

np.save(
    SUB_DIR / "final_pred_combo_te_v1_seed1234.npy",
    final_pred_seed1234
)

print("\nseed1234 저장 완료")
print(submission_seed1234[pred_col].describe())


seed1234 저장 완료
count    90067.000000
mean         0.500006
std          0.288347
min          0.000048
25%          0.250761
50%          0.499865
75%          0.749563
max          0.999984
Name: probability, dtype: float64


In [18]:
seed42_oof = np.load("../oof_preds/combo_te_s10/final_combo_rank.npy")
seed2024_oof = np.load("../oof_preds/combo_te_v1_s10_seed2024/final_oof_seed2024.npy")
seed777_oof = np.load("../oof_preds/combo_te_v1_s10_seed777/final_oof_seed777.npy")
seed999_oof = np.load("../oof_preds/combo_te_v1_s10_seed999/final_oof_seed999.npy")
seed1234_oof = np.load("../oof_preds/combo_te_v1_s10_seed1234/final_oof_seed1234.npy")

avg_3 = (seed42_oof + seed2024_oof + seed777_oof) / 3
avg_4 = (seed42_oof + seed2024_oof + seed777_oof + seed999_oof) / 4
avg_5 = (seed42_oof + seed2024_oof + seed777_oof + seed999_oof+seed999_oof) / 5
print("seed1234:", roc_auc_score(y_stack, seed1234_oof))
print("avg 3:", roc_auc_score(y_stack, avg_3))
print("avg 4:", roc_auc_score(y_stack, avg_4))
print("avg 5:", roc_auc_score(y_stack, avg_5))

seed1234: 0.7403779062281844
avg 3: 0.7407465066345057
avg 4: 0.7407705074934163
avg 5: 0.7407492598087494


In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("/mnt/c/dev/my_ml_project")

SUB_DIR = PROJECT_ROOT / "submissions" / "seed_ensemble"
SUB_DIR.mkdir(parents=True, exist_ok=True)

seed42_test = np.load(PROJECT_ROOT / "submissions" / "preds" / "final_aggressive_pred_lb_0_74172.npy")
seed2024_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed2024.npy")
seed777_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed777.npy")
seed999_test = np.load(PROJECT_ROOT/"submissions"/"seed_ensemble" / "final_pred_combo_te_v1_seed999.npy")
seed1234_test = np.load(PROJECT_ROOT / "submissions" / "seed_ensemble" / "final_pred_combo_te_v1_seed1234.npy")

final_pred_seed42_2024_777_999_1234_avg = (
    seed42_test +
    seed2024_test +
    seed777_test +
    seed1234_test+
    seed999_test
) / 5

submission = pd.read_csv(PROJECT_ROOT / "data" / "sample_submission.csv")
pred_col = submission.columns[-1]

submission[pred_col] = final_pred_seed42_2024_777_999_1234_avg

out_csv = SUB_DIR / "submission_combo_te_v1_rank_seed42_2024_777_999_1234_avg.csv"
out_npy = SUB_DIR / "final_pred_combo_te_v1_seed42_2024_777_1234_999_avg.npy"

submission.to_csv(out_csv, index=False)
np.save(out_npy, final_pred_seed42_2024_777_999_1234_avg)

print("saved csv:", out_csv)
print("saved npy:", out_npy)
print(submission[pred_col].describe())

saved csv: /mnt/c/dev/my_ml_project/submissions/seed_ensemble/submission_combo_te_v1_rank_seed42_2024_777_999_1234_avg.csv
saved npy: /mnt/c/dev/my_ml_project/submissions/seed_ensemble/final_pred_combo_te_v1_seed42_2024_777_1234_999_avg.npy
count    90067.000000
mean         0.500006
std          0.288311
min          0.000029
25%          0.250853
50%          0.500042
75%          0.749605
max          0.999987
Name: probability, dtype: float64
